# 2.0 - SQL Stream Translation Support Browser

Translating the support browser logic into a reproducible notebook workflow using the cleaned datasets


In [22]:
from pathlib import Path
import pandas as pd
from IPython.display import display
import ipywidgets as widgets

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 250)
pd.set_option("display.float_format", "{:,.2f}".format)


In [23]:
PROJECT_DIR = Path(".")
clean_data_dir = PROJECT_DIR / "outputs" / "cleaned_data"
support_cleaned_file = clean_data_dir / "support_cleaned.csv"
gl_accounts_cleaned_file = clean_data_dir / "gl_accounts_cleaned.csv"
gl_entries_cleaned_file = clean_data_dir / "gl_entries_cleaned.csv"
YEAR = 2026
START_DATE = pd.Timestamp(f"{YEAR}-01-01")
END_DATE = pd.Timestamp(f"{YEAR + 1}-01-01")
EXCLUDED_SUPPLIER_CODES = {"E005GB", "E059"}
MEDIUM_RISK_THRESHOLD = 2500.0
HIGH_RISK_THRESHOLD = 10000.0
print("Project folder:", PROJECT_DIR.resolve())
print("Cleaned support:", support_cleaned_file.exists(), "cleaned GL accounts:", gl_accounts_cleaned_file.exists(), "cleaned GL entries:", gl_entries_cleaned_file.exists())


Project folder: \\bpfile\product management\Product Management\(4) Cathal Heaney\Uni\8030
Cleaned support: True cleaned GL accounts: True cleaned GL entries: True


## Helper Functions


In [24]:
def read_cleaned_dataset(path):
    if not path.exists():
        raise FileNotFoundError(f"Cleaned dataset not found: {path}. Run 0.2 - Data Cleaning first.")
    return pd.read_csv(path, low_memory=False)


def supplier_group_code(code):
    if pd.isna(code): return ""
    code = str(code).strip()
    if code in {"L810", "L812"}: return "L810/L812"
    if code in {"E907", "E907GB"}: return "E907/E907GB"
    return code


def supplier_group_name(code, name):
    group = supplier_group_code(code)
    if group == "L810/L812": return "Supplier14"
    if group == "E907/E907GB": return "Supplier21"
    return "" if pd.isna(name) else str(name).strip()


def supplier_key(series):
    return series.astype("string").str.strip().str.upper().str.replace(" ", "", regex=False)


def fmt_money(value):
    value = 0.0 if pd.isna(value) else float(value)
    sign = "-" if value < 0 else ""
    return f"{sign}GBP {abs(value):,.2f}"


def risk_band(outstanding):
    outstanding = 0.0 if pd.isna(outstanding) else float(outstanding)
    if outstanding >= HIGH_RISK_THRESHOLD: return "High"
    if outstanding >= MEDIUM_RISK_THRESHOLD: return "Medium"
    if outstanding > 0: return "Low"
    return "Clear"


def show_df(df, money_cols=None, int_cols=None, date_cols=None, max_rows=350):
    money_cols, int_cols, date_cols = money_cols or [], int_cols or [], date_cols or []
    if df is None or df.empty:
        print("No rows for this view."); return
    out = df.head(max_rows).copy()
    for col in money_cols:
        if col in out: out[col] = out[col].map(fmt_money)
    for col in int_cols:
        if col in out: out[col] = out[col].fillna(0).map(lambda x: f"{float(x):,.0f}")
    for col in date_cols:
        if col in out: out[col] = pd.to_datetime(out[col], errors="coerce").dt.strftime("%Y-%m-%d").fillna("")
    display(out)
    if len(df) > max_rows: print(f"Showing first {max_rows:,} of {len(df):,} rows.")


## Load Data


In [25]:
def load_support_data():
    usecols = ["Date", "claim_date", "Customer", "Customer_Name", "Product", "Product_ManufacturerProductCode", "Product_Category", "Contract_Number", "SourceTransactionType", "Contract_D_ContractNumber", "SalesInvoice_Number", "SalesDeliveryNote_Number", "Status", "UnitClaimAmount", "SalesDeliveryNoteLine_Quantity", "SalesInvoiceLine_Quantity", "SalesDeliveryNote_NetAmountLessDiscountBase", "Contract_Description", "Contract_Expression", "Contract_ValidFrom", "Contract_ValidTo", "Contract_Supplier_Code", "Contract_Supplier_Name", "credit_due_value"]
    df = read_cleaned_dataset(support_cleaned_file)
    df = df[[col for col in usecols if col in df.columns]]
    df["Date"] = pd.to_datetime(df["claim_date"] if "claim_date" in df.columns else df["Date"], errors="coerce")
    df = df[df["Date"].ge(START_DATE) & df["Date"].lt(END_DATE) & df["SourceTransactionType"].eq("SL/Del") & ~df["Contract_Supplier_Code"].isin(EXCLUDED_SUPPLIER_CODES)].copy()
    df["SupplierCode"] = df["Contract_Supplier_Code"].map(supplier_group_code)
    df["SupplierName"] = [supplier_group_name(c, n) for c, n in zip(df["Contract_Supplier_Code"], df["Contract_Supplier_Name"])]
    df["SupplierKey"] = supplier_key(df["SupplierName"])
    df["Quantity"] = pd.to_numeric(df["SalesDeliveryNoteLine_Quantity"], errors="coerce").fillna(pd.to_numeric(df["SalesInvoiceLine_Quantity"], errors="coerce")).fillna(0)
    df["TotalSupportOwed"] = pd.to_numeric(df["credit_due_value"], errors="coerce").fillna(0)
    df["TotalNetAmountLessDiscountBase"] = pd.to_numeric(df["SalesDeliveryNote_NetAmountLessDiscountBase"], errors="coerce").fillna(0)
    df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()
    return df


def load_ledger_data(support):
    accounts_raw = read_cleaned_dataset(gl_accounts_cleaned_file)
    entries_raw = read_cleaned_dataset(gl_entries_cleaned_file)
    accounts = pd.DataFrame({
        "GLAccountCode": accounts_raw.get("gl_account_code", accounts_raw.get("Code")),
        "GLAccountDescription": accounts_raw.get("gl_account_description", accounts_raw.get("Description")),
        "SupplierKey": accounts_raw.get("gl_supplier_key", pd.Series(pd.NA, index=accounts_raw.index)),
        "CurrentBalance": pd.to_numeric(accounts_raw.get("gl_current_balance", accounts_raw.get("CurrentBalance")), errors="coerce"),
    })
    entries = pd.DataFrame({
        "GLAccountCode": entries_raw.get("gl_account_code", entries_raw.get("GL_Account_Code")),
        "GLAccountDescription": entries_raw.get("gl_account_description", entries_raw.get("Description")),
        "SupplierKey": entries_raw.get("gl_supplier_key", pd.Series(pd.NA, index=entries_raw.index)),
        "LedgerDate": pd.to_datetime(entries_raw.get("gl_entry_line_date", entries_raw.get("Entry_Line_Date")), errors="coerce"),
        "CreditAmount": pd.to_numeric(entries_raw.get("gl_credit_amount", entries_raw.get("Credit_Amount")), errors="coerce").fillna(0),
        "RunningBalance": pd.to_numeric(entries_raw.get("Running_Balance"), errors="coerce"),
        "TotalBalance": pd.to_numeric(entries_raw.get("Total_Balance"), errors="coerce"),
    })
    account_cols = ["GLAccountCode", "SupplierKey", "CurrentBalance"]
    ledger = entries.merge(accounts[account_cols].drop_duplicates("GLAccountCode"), on=["GLAccountCode", "SupplierKey"], how="left")
    supplier_lookup = support[["SupplierKey", "SupplierCode", "SupplierName"]].drop_duplicates("SupplierKey")
    ledger = ledger.merge(supplier_lookup, on="SupplierKey", how="left")
    ledger["SupplierCode"] = ledger["SupplierCode"].fillna("")
    ledger["SupplierName"] = ledger["SupplierName"].fillna(ledger["SupplierKey"])
    ledger = ledger[ledger["LedgerDate"].ge(START_DATE) & ledger["LedgerDate"].lt(END_DATE)].copy()
    ledger["Month"] = ledger["LedgerDate"].dt.to_period("M").dt.to_timestamp()
    return ledger


def build_summary(support, ledger):
    names = support.groupby("SupplierCode", as_index=False).agg(SupplierName=("SupplierName", "first"), LinkedSupplierCodes=("Contract_Supplier_Code", lambda x: ", ".join(sorted(set(map(str, x.dropna()))))))
    sup = support.groupby("SupplierCode", as_index=False).agg(LineCount=("Date", "size"), ContractCount=("Contract_D_ContractNumber", "nunique"), CustomerCount=("Customer", "nunique"), ProductCount=("Product", "nunique"), TotalQuantity=("Quantity", "sum"), TotalSupportOwed=("TotalSupportOwed", "sum"), FirstSupportDate=("Date", "min"), LastSupportDate=("Date", "max"))
    led = ledger[ledger["SupplierCode"].ne("")].groupby("SupplierCode", as_index=False).agg(LedgerPaid=("CreditAmount", "sum"), LatestLedgerDate=("LedgerDate", "max"), LedgerLineCount=("LedgerDate", "size"), GLAccounts=("GLAccountCode", lambda x: ", ".join(sorted(set(map(str, x.dropna()))))))
    summary = names.merge(sup, on="SupplierCode", how="left").merge(led, on="SupplierCode", how="left")
    summary["LedgerPaid"] = summary["LedgerPaid"].fillna(0)
    summary["Outstanding"] = summary["TotalSupportOwed"].fillna(0) - summary["LedgerPaid"]
    summary["RiskBand"] = summary["Outstanding"].map(risk_band)
    return summary.sort_values(["Outstanding", "TotalSupportOwed"], ascending=False)


def load_or_build():
    support = load_support_data()
    ledger = load_ledger_data(support)
    summary = build_summary(support, ledger)
    return support, ledger, summary


support, ledger, supplier_summary = load_or_build()
print("Support rows:", len(support), "Ledger rows:", len(ledger), "Supplier groups:", len(supplier_summary))
display(supplier_summary.head(10))


Support rows: 13464 Ledger rows: 100 Supplier groups: 19


,SupplierCode,SupplierName,LinkedSupplierCodes,LineCount,ContractCount,CustomerCount,ProductCount,TotalQuantity,TotalSupportOwed,FirstSupportDate,LastSupportDate,LedgerPaid,LatestLedgerDate,LedgerLineCount,GLAccounts,Outstanding,RiskBand
2,E070,Supplier19,E070,444,13,14,127,1144,"111,402.22",2026-01-05,2026-06-18,0.00,NaT,NaN,NaN,"111,402.22",High
3,E071,Supplier13,E071,699,29,106,21,1152,"257,950.37",2026-01-05,2026-06-18,"179,305.04",2026-03-07,82.00,01-85001,"78,645.33",High
17,U102,Supplier4,U102,197,6,62,16,251,"67,749.74",2026-01-05,2026-06-18,0.00,NaT,NaN,NaN,"67,749.74",High
18,U106,Supplier1,U106,7542,96,181,198,48173,"41,075.68",2026-01-05,2026-06-18,0.00,NaT,NaN,NaN,"41,075.68",High
8,E555,Supplier18,E555,1512,36,35,119,6082,"49,246.71",2026-01-05,2026-06-18,"9,662.89",2026-10-06,2.00,01-85079,"39,583.82",High
10,E907/E907GB,Supplier21,"E907, E907GB",138,7,55,24,160,"34,462.00",2026-01-05,2026-06-17,"17,397.00",2026-08-04,5.00,01-85185,"17,065.00",High
4,E102,Supplier7,E102,311,2,113,14,676,"7,571.96",2026-01-05,2026-06-18,0.00,NaT,NaN,NaN,"7,571.96",Medium
11,E948,Supplier20,E948,255,11,11,97,2155,"5,604.23",2026-01-05,2026-06-18,76.22,2026-07-04,1.00,01-85115,"5,528.01",Medium
13,L810/L812,Supplier14,"L810, L812",899,0,17,139,32470,"7,208.41",2026-01-05,2026-06-18,"2,002.26",2026-08-07,2.00,01-85137,"5,206.15",Medium
5,E175,Supplier8,E175,824,3,230,10,1634,"5,647.20",2026-01-05,2026-06-18,"2,459.31",2026-04-06,1.00,01-85036,"3,187.89",Medium


## Detail Functions


In [26]:
def contract_view(code):
    df = support[support["SupplierCode"].eq(code)]
    return df.groupby(["Contract_Number","Contract_D_ContractNumber","Contract_Description","Contract_Expression","Contract_ValidFrom","Contract_ValidTo"], dropna=False, as_index=False).agg(LineCount=("Date","size"), CustomerCount=("Customer","nunique"), ProductCount=("Product","nunique"), TotalQuantity=("Quantity","sum"), TotalSupportOwed=("TotalSupportOwed","sum")).sort_values("Contract_D_ContractNumber")

def customer_view(code):
    df = support[support["SupplierCode"].eq(code)]
    return df.groupby(["Customer","Customer_Name"], dropna=False, as_index=False).agg(LineCount=("Date","size"), ContractCount=("Contract_D_ContractNumber","nunique"), ProductCount=("Product","nunique"), TotalQuantity=("Quantity","sum"), TotalNetAmountLessDiscountBase=("TotalNetAmountLessDiscountBase","sum"), TotalSupportOwed=("TotalSupportOwed","sum")).sort_values("TotalSupportOwed", ascending=False)

def product_view(code):
    df = support[support["SupplierCode"].eq(code)]
    return df.groupby(["Product","Product_ManufacturerProductCode","Product_Category"], dropna=False, as_index=False).agg(LineCount=("Date","size"), ContractCount=("Contract_D_ContractNumber","nunique"), CustomerCount=("Customer","nunique"), TotalQuantity=("Quantity","sum"), TotalNetAmountLessDiscountBase=("TotalNetAmountLessDiscountBase","sum"), TotalSupportOwed=("TotalSupportOwed","sum")).sort_values(["Product_Category","Product"])

def monthly_view(code):
    sm = support[support["SupplierCode"].eq(code)].groupby("Month", as_index=False).agg(SupportLines=("Date","size"), Customers=("Customer","nunique"), Products=("Product","nunique"), Quantity=("Quantity","sum"), SupportOwed=("TotalSupportOwed","sum"))
    lm = ledger[ledger["SupplierCode"].eq(code)].groupby("Month", as_index=False).agg(LedgerLines=("LedgerDate","size"), LedgerPaid=("CreditAmount","sum"), LatestLedgerDate=("LedgerDate","max"))
    months = pd.DataFrame({"Month": pd.date_range(START_DATE, END_DATE - pd.offsets.MonthBegin(1), freq="MS")})
    out = months.merge(sm, on="Month", how="left").merge(lm, on="Month", how="left")
    for col in ["SupportLines","Customers","Products","Quantity","SupportOwed","LedgerLines","LedgerPaid"]: out[col] = out[col].fillna(0)
    out["OutstandingMovement"] = out["SupportOwed"] - out["LedgerPaid"]
    out["CumulativeOutstanding"] = out["OutstandingMovement"].cumsum()
    out["Month"] = out["Month"].dt.strftime("%Y-%m")
    return out

def ledger_view(code):
    cols = [c for c in ["LedgerDate","GLAccountCode","GLAccountDescription","Description","CreditAmount","RunningBalance"] if c in ledger]
    return ledger[ledger["SupplierCode"].eq(code)][cols].sort_values("LedgerDate", ascending=False)

def support_lines_view(code):
    cols = ["Date", "Customer", "Customer_Name", "Product", "Product_ManufacturerProductCode", "Product_Category", "Contract_D_ContractNumber", "Contract_Description", "Quantity", "UnitClaimAmount", "TotalSupportOwed", "TotalNetAmountLessDiscountBase", "SalesDeliveryNote_Number", "SalesInvoice_Number", "Status"]
    cols = [col for col in cols if col in support.columns]
    return support[support["SupplierCode"].eq(code)][cols].sort_values("Date", ascending=False)


In [27]:
display(supplier_summary.head(10))


,SupplierCode,SupplierName,LinkedSupplierCodes,LineCount,ContractCount,CustomerCount,ProductCount,TotalQuantity,TotalSupportOwed,FirstSupportDate,LastSupportDate,LedgerPaid,LatestLedgerDate,LedgerLineCount,GLAccounts,Outstanding,RiskBand
2,E070,Supplier19,E070,444,13,14,127,1144,"111,402.22",2026-01-05,2026-06-18,0.00,NaT,NaN,NaN,"111,402.22",High
3,E071,Supplier13,E071,699,29,106,21,1152,"257,950.37",2026-01-05,2026-06-18,"179,305.04",2026-03-07,82.00,01-85001,"78,645.33",High
17,U102,Supplier4,U102,197,6,62,16,251,"67,749.74",2026-01-05,2026-06-18,0.00,NaT,NaN,NaN,"67,749.74",High
18,U106,Supplier1,U106,7542,96,181,198,48173,"41,075.68",2026-01-05,2026-06-18,0.00,NaT,NaN,NaN,"41,075.68",High
8,E555,Supplier18,E555,1512,36,35,119,6082,"49,246.71",2026-01-05,2026-06-18,"9,662.89",2026-10-06,2.00,01-85079,"39,583.82",High
10,E907/E907GB,Supplier21,"E907, E907GB",138,7,55,24,160,"34,462.00",2026-01-05,2026-06-17,"17,397.00",2026-08-04,5.00,01-85185,"17,065.00",High
4,E102,Supplier7,E102,311,2,113,14,676,"7,571.96",2026-01-05,2026-06-18,0.00,NaT,NaN,NaN,"7,571.96",Medium
11,E948,Supplier20,E948,255,11,11,97,2155,"5,604.23",2026-01-05,2026-06-18,76.22,2026-07-04,1.00,01-85115,"5,528.01",Medium
13,L810/L812,Supplier14,"L810, L812",899,0,17,139,32470,"7,208.41",2026-01-05,2026-06-18,"2,002.26",2026-08-07,2.00,01-85137,"5,206.15",Medium
5,E175,Supplier8,E175,824,3,230,10,1634,"5,647.20",2026-01-05,2026-06-18,"2,459.31",2026-04-06,1.00,01-85036,"3,187.89",Medium


## Notebook Display


In [28]:
def display_supplier_drilldown(code):
    print("Selected supplier:", code)
    display(supplier_summary[supplier_summary["SupplierCode"].eq(code)])
    display(monthly_view(code))
    display(contract_view(code).head(10))
    display(customer_view(code).head(10))
    display(product_view(code).head(10))
    display(ledger_view(code).head(10))
    display(support_lines_view(code).head(10))

display_supplier_drilldown("E070")


Selected supplier: E070


,SupplierCode,SupplierName,LinkedSupplierCodes,LineCount,ContractCount,CustomerCount,ProductCount,TotalQuantity,TotalSupportOwed,FirstSupportDate,LastSupportDate,LedgerPaid,LatestLedgerDate,LedgerLineCount,GLAccounts,Outstanding,RiskBand
2,E070,Supplier19,E070,444,13,14,127,1144,"111,402.22",2026-01-05,2026-06-18,0.00,NaT,NaN,NaN,"111,402.22",High


,Month,SupportLines,Customers,Products,Quantity,SupportOwed,LedgerLines,LedgerPaid,LatestLedgerDate,OutstandingMovement,CumulativeOutstanding
0,2026-01,64.00,4.00,45.00,141.00,"12,558.40",0.00,0.00,NaT,"12,558.40","12,558.40"
1,2026-02,62.00,6.00,42.00,347.00,"20,034.20",0.00,0.00,NaT,"20,034.20","32,592.59"
2,2026-03,97.00,4.00,53.00,164.00,"22,141.02",0.00,0.00,NaT,"22,141.02","54,733.62"
3,2026-04,110.00,3.00,47.00,209.00,"24,179.93",0.00,0.00,NaT,"24,179.93","78,913.55"
4,2026-05,77.00,8.00,52.00,228.00,"26,818.30",0.00,0.00,NaT,"26,818.30","105,731.85"
5,2026-06,34.00,7.00,29.00,55.00,"5,670.38",0.00,0.00,NaT,"5,670.38","111,402.22"
6,2026-07,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaT,0.00,"111,402.22"
7,2026-08,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaT,0.00,"111,402.22"
8,2026-09,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaT,0.00,"111,402.22"
9,2026-10,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaT,0.00,"111,402.22"


,Contract_Number,Contract_D_ContractNumber,Contract_Description,Contract_Expression,Contract_ValidFrom,Contract_ValidTo,LineCount,CustomerCount,ProductCount,TotalQuantity,TotalSupportOwed
12,"1,555.00",6147690,Stelrad Compact With Style - KANE GROUP,[D_InvoiceCost]*0.3100,01/01/0001,01/01/0001,1,1,1,2,349.36
0,453.00,6152315,Stelrad Elite - NIHE,[D_InvoiceCost]*0.2468,01/01/0001,01/01/0001,219,1,58,248,"4,730.63"
2,521.00,6154489,Stelrad Compact With Style - FG Mechanical,[D_InvoiceCost]*0.4371,01/01/0001,01/01/0001,13,2,6,54,"9,300.27"
13,"1,606.00",6154809,Stelrad LST Standard - Devlin Mech.,[D_InvoiceCost]*0.4500,01/01/0001,01/01/0001,13,1,8,15,"2,899.58"
1,456.00,6154920,Stelrad Compact - Grove Mechanical,[D_InvoiceCost]*0.1548,01/01/0001,01/01/0001,6,1,2,152,"1,094.60"
3,715.00,6155141,Stelrad Compact with Style - IRWIN M & E,[D_InvoiceCost]*0.5267,01/01/0001,01/01/0001,87,1,12,524,"56,481.10"
5,917.00,6156397,Stelrad Planar - K&M Tech,[D_InvoiceCost]*0.3567,01/01/0001,01/01/0001,22,1,9,25,"7,790.97"
6,918.00,6156397,Stelrad LST Standard / iPlus - K&M Tech,[D_InvoiceCost]*0.4253,01/01/0001,01/01/0001,4,1,2,4,"1,261.13"
8,935.00,6156475,Stelrad LST STD / I Plus - Victoria Mech.,[D_InvoiceCost]*0.4613,01/01/0001,01/01/0001,42,1,12,74,"19,091.84"
11,997.00,6156546,Stelrad LST STD/iPlus - FG Mechanical (GENERIC),[D_InvoiceCost]*0.4387,01/01/0001,01/01/0001,3,1,3,3,691.66


,Customer,Customer_Name,LineCount,ContractCount,ProductCount,TotalQuantity,TotalNetAmountLessDiscountBase,TotalSupportOwed
8,IRW100GB,Irwin M&E Limited,87,1,12,524,"302,146.35","56,481.10"
13,VIC480,Victoria Mechanical Services Ltd,42,1,12,74,"301,624.70","19,091.84"
10,KMT100GB,K&M Technical Services Limited,26,1,11,29,"154,011.05","9,052.10"
4,FGP100GB,Fg Mechanical Building Services Ltd,12,1,5,51,"29,300.75","8,732.48"
7,HAA540,H & A Mechanical Services Ltd.,219,1,58,248,"413,816.43","4,730.63"
2,DEV528,Devlin Mechanical Limited,29,2,17,34,"81,761.91","4,575.56"
1,AND490,David Anderson (Plumbing & Heating Services) Ltd,12,1,5,12,"66,859.70","4,037.09"
3,DPH100,Donnelly Plumbing And Heating,2,1,2,7,"3,241.64","1,211.56"
6,GRO455,Grove Mechanical Services Ltd.,6,1,2,152,"21,853.14","1,094.60"
12,PAP500,Paul Toner & Paul Mccullagh T/A P&P Mechanical...,3,1,3,3,"3,216.27",691.66


,Product,Product_ManufacturerProductCode,Product_Category,LineCount,ContractCount,CustomerCount,TotalQuantity,TotalNetAmountLessDiscountBase,TotalSupportOwed
69,STEL8432,8432,STEL1,1,1,1,3,224.19,31.41
70,STEL8443,8443,STEL1,1,1,1,1,224.19,13.19
71,STEL8444,8444,STEL1,1,1,1,1,"3,055.33",19.96
72,STEL8461,8461,STEL1,2,1,1,2,"3,090.50",10.06
73,STEL8464,8464,STEL1,1,1,1,1,"1,668.97",7.47
74,STEL8466,8466,STEL1,2,1,1,2,"3,215.37",18.53
75,STEL8468,8468,STEL1,2,1,1,2,"3,248.02",22.67
76,STEL8469,8469,STEL1,2,1,1,3,"4,499.88",37.10
77,STEL8470,8470,STEL1,2,1,1,2,"3,248.02",28.19
78,STEL8471,8471,STEL1,1,1,1,1,"1,546.40",15.69


,LedgerDate,GLAccountCode,GLAccountDescription,CreditAmount,RunningBalance


,Date,Customer,Customer_Name,Product,Product_ManufacturerProductCode,Product_Category,Contract_D_ContractNumber,Contract_Description,Quantity,UnitClaimAmount,TotalSupportOwed,TotalNetAmountLessDiscountBase,SalesDeliveryNote_Number,SalesInvoice_Number,Status
73238,2026-06-18,ADA090,Adair Building Services Ltd_x000D_\n,STEL163025C,163025C,STEL9C,6156953,Stelrad - Stelrad Column - Adair Building Serv...,1,190.20,190.20,"1,322.69",SO0381206/1,NaN,Claimed
73237,2026-06-18,ADA090,Adair Building Services Ltd_x000D_\n,STEL163024C,163024C,STEL9C,6156953,Stelrad - Stelrad Column - Adair Building Serv...,2,156.97,313.94,"1,322.69",SO0381206/1,NaN,Claimed
73046,2026-06-16,HAA540,H & A Mechanical Services Ltd.,STEL8571,8571,STEL1,6152315,Stelrad Elite - NIHE,1,23.41,23.41,"2,571.92",SO0389269/1,SI00429027,Claimed
73045,2026-06-16,HAA540,H & A Mechanical Services Ltd.,STEL8574,8574,STEL1,6152315,Stelrad Elite - NIHE,2,34.30,68.60,"2,571.92",SO0389269/1,SI00429027,Claimed
73044,2026-06-16,HAA540,H & A Mechanical Services Ltd.,STEL8537,8537,STEL1,6152315,Stelrad Elite - NIHE,2,18.45,36.90,"2,571.92",SO0389269/1,SI00429027,Claimed
73043,2026-06-16,HAA540,H & A Mechanical Services Ltd.,STEL8469,8469,STEL1,6152315,Stelrad Elite - NIHE,2,12.56,25.13,"2,571.92",SO0389269/1,SI00429027,Claimed
72569,2026-06-11,PAP500,Paul Toner & Paul Mccullagh T/A P&P Mechanical...,STEL145045,145045,STEL16,6156546,Stelrad LST STD/iPlus - FG Mechanical (GENERIC),1,302.41,302.41,"1,072.09",SO0387622/2,SI00426857,Claimed
72568,2026-06-11,PAP500,Paul Toner & Paul Mccullagh T/A P&P Mechanical...,STEL145058,145058,STEL16,6156546,Stelrad LST STD/iPlus - FG Mechanical (GENERIC),1,198.56,198.56,"1,072.09",SO0387622/2,SI00426857,Claimed
72567,2026-06-11,PAP500,Paul Toner & Paul Mccullagh T/A P&P Mechanical...,STEL145042,145042,STEL16,6156546,Stelrad LST STD/iPlus - FG Mechanical (GENERIC),1,190.69,190.69,"1,072.09",SO0387622/2,SI00426857,Claimed
72470,2026-06-10,HAA540,H & A Mechanical Services Ltd.,STEL8571,8571,STEL1,6152315,Stelrad Elite - NIHE,2,23.41,46.81,"1,764.34",SO0385901/1,SI00426104,Claimed
